In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "chromadb>=1.5.7",
    "openai>=2.32.0",
    "python-dotenv>=1.2.2",
])



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


0

# RAG and tool calling walkthrough

This notebook is meant for a live demo. Run one cell at a time and narrate the transition from plain documents, to retrieval, to grounded generation, to tool calling.

The core mental model:

1. RAG is not magic memory. It is search plus a prompt.
2. The retrieval step chooses relevant external context.
3. The generation step asks the model to answer from that context.
4. Tool calling lets the model decide when to run code or fetch context before answering.


## Setup

Before running the model-backed cells, start LM Studio's local server and load:

- Chat model: `qwen/qwen3.5-9b`
- Embedding model: `text-embedding-qwen3-embedding-4b`

The notebook lives at the demo root and reads documents from `documents/`. It loads `.env` from the demo root if present, then from `rag_demo/.env`.

In [2]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "documents").exists() and (PROJECT_DIR.parent / "documents").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DOCS_DIR = PROJECT_DIR / "documents"
load_dotenv(PROJECT_DIR / ".env")
load_dotenv(PROJECT_DIR / "rag_demo" / ".env")


@dataclass(frozen=True)
class Config:
    openai_base_url: str = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1")
    openai_api_key: str = os.getenv("OPENAI_API_KEY", "lm-studio")
    chat_model: str = os.getenv("CHAT_MODEL", "qwen/qwen3.5-9b")
    embedding_model: str = os.getenv("EMBEDDING_MODEL", "text-embedding-qwen3-embedding-4b")
    top_k: int = int(os.getenv("TOP_K", "3"))


cfg = Config()
client = OpenAI(base_url=cfg.openai_base_url, api_key=cfg.openai_api_key)

print(f"Project directory : {PROJECT_DIR}")
print(f"Documents         : {DOCS_DIR}")
print(f"LM Studio URL     : {cfg.openai_base_url}")
print(f"Chat model        : {cfg.chat_model}")
print(f"Embedding model   : {cfg.embedding_model}")
print(f"Top-K             : {cfg.top_k}")

Project directory : /Users/dinu/dev/mobile-and-embedded-computing/other-presentations/devtalks-roundtable/demo
Documents         : /Users/dinu/dev/mobile-and-embedded-computing/other-presentations/devtalks-roundtable/demo/documents
LM Studio URL     : http://127.0.0.1:1234/v1
Chat model        : qwen/qwen3.5-9b
Embedding model   : text-embedding-qwen3-embedding-4b
Top-K             : 3


In [4]:
models = client.models.list()
print("Models visible to the OpenAI-compatible endpoint:")
for model in models.data:
    print("-", model.id)

Models visible to the OpenAI-compatible endpoint:
- google/gemma-4-26b-a4b
- qwen/qwen3.6-27b
- qwen/qwen3-coder-30b
- google/gemma-4-e4b
- qwen/qwen3.5-9b
- text-embedding-nomic-embed-text-v1.5
- qwen/qwen3-4b-2507
- text-embedding-qwen3-embedding-4b
- qwen/qwen3-14b


## 1. Load the knowledge base

This demo uses one markdown file per chunk so the retrieval step is easy to see.

In [5]:
def load_markdown_files(directory: Path) -> list[dict[str, str]]:
    docs = []
    for path in sorted(directory.glob("*.md")):
        docs.append({"id": path.stem, "source": path.name, "text": path.read_text(encoding="utf-8")})
    if not docs:
        raise FileNotFoundError(f"No markdown files found in {directory}")
    return docs


docs = load_markdown_files(DOCS_DIR)
print(f"Loaded {len(docs)} documents")
for doc in docs:
    first_line = doc["text"].splitlines()[0]
    print(f"- {doc['source']}: {first_line}")

Loaded 6 documents
- devtalks_conference.md: # DevTalks
- lm_studio.md: # LM Studio local server
- mcp_overview.md: # Model Context Protocol (MCP)
- mcp_plus_rag.md: # MCP + RAG together
- qwen_models.md: # Qwen models used in this demo
- rag_overview.md: # Retrieval-Augmented Generation (RAG)


In [6]:
# Pick one document and show that the source text is plain, inspectable data.
display(Markdown(docs[0]["text"]))

# DevTalks

DevTalks is Romania's largest developer conference, bringing together
software engineers, tech leaders, and innovators. It covers talks,
workshops, and networking opportunities across a wide range of topics
including AI, cloud, mobile, and software architecture.

The official site is
[devtalks.ro](https://www.devtalks.ro/).


## 2. Turn text into vectors

An embedding is a list of numbers. Similar meanings should land near each other in vector space. The important rule for RAG: use the same embedding model for documents and questions.

In [7]:
sample_embedding = client.embeddings.create(
    model=cfg.embedding_model,
    input="What is retrieval augmented generation?",
).data[0].embedding

print(f"Embedding dimensions: {len(sample_embedding)}")
print("First 8 values:", [round(x, 4) for x in sample_embedding[:8]])

Embedding dimensions: 2560
First 8 values: [-0.0005, 0.0113, -0.0007, 0.0324, -0.0025, 0.1011, 0.0371, 0.0078]


In [8]:
class OpenAIEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, openai_client: OpenAI, model: str) -> None:
        self._client = openai_client
        self._model = model

    def __call__(self, inputs: Documents) -> Embeddings:
        response = self._client.embeddings.create(model=self._model, input=list(inputs))
        return [item.embedding for item in response.data]


embedder = OpenAIEmbeddingFunction(client, cfg.embedding_model)
chroma = chromadb.EphemeralClient()
collection = chroma.get_or_create_collection(
    name="devtalks_notebook_rag_demo",
    embedding_function=embedder,
)

collection.upsert(
    ids=[doc["id"] for doc in docs],
    documents=[doc["text"] for doc in docs],
    metadatas=[{"source": doc["source"]} for doc in docs],
)

print(f"Indexed {collection.count()} documents in an in-memory Chroma collection")

Indexed 6 documents in an in-memory Chroma collection


## 3. Compare embedding model sizes

Embedding model size affects retrieval quality, latency, and memory. A larger model often separates subtle meanings better, but it costs more to run. A smaller model is faster and lighter, but may return a different ranking for ambiguous questions.

This comparison indexes the same documents twice: once with the 4B embedding model and once with `qwen3-0.6b-text-embedding`. The chat model stays unchanged; only the retrieval model changes.

In [11]:
import time

EMBEDDING_MODELS_TO_COMPARE = [
    cfg.embedding_model,
    os.getenv("SMALL_EMBEDDING_MODEL", "qwen3-0.6b-text-embedding"),
]


def safe_collection_name(model: str) -> str:
    safe = "".join(ch if ch.isalnum() else "_" for ch in model.lower())
    return f"devtalks_embed_compare_{safe}"[:63]


def build_collection_for_model(model: str):
    start = time.perf_counter()
    test_embedding = client.embeddings.create(
        model=model,
        input="What is retrieval augmented generation?",
    ).data[0].embedding

    model_chroma = chromadb.EphemeralClient()
    model_collection = model_chroma.get_or_create_collection(
        name=safe_collection_name(model),
        embedding_function=OpenAIEmbeddingFunction(client, model),
    )
    model_collection.upsert(
        ids=[doc["id"] for doc in docs],
        documents=[doc["text"] for doc in docs],
        metadatas=[{"source": doc["source"]} for doc in docs],
    )
    elapsed = time.perf_counter() - start
    return {
        "model": model,
        "collection": model_collection,
        "dimensions": len(test_embedding),
        "index_seconds": elapsed,
    }


embedding_comparison = []
for model in EMBEDDING_MODELS_TO_COMPARE:
    try:
        result = build_collection_for_model(model)
        embedding_comparison.append(result)
        print(
            f"{model}: {result['dimensions']} dimensions, "
            f"indexed {len(docs)} docs in {result['index_seconds']:.2f}s"
        )
    except Exception as exc:
        print(f"{model}: unavailable or failed ({exc})")

text-embedding-qwen3-embedding-4b: 2560 dimensions, indexed 6 docs in 4.05s
qwen3-0.6b-text-embedding: 1024 dimensions, indexed 6 docs in 2.71s


In [ ]:
comparison_questions = [
    "What are the three stages of RAG?",
    "How does MCP change the RAG pipeline?",
    "What model serves embeddings in this demo?",
]


def retrieve_from_collection(model_collection, question: str, k: int = 3):
    start = time.perf_counter()
    result = model_collection.query(query_texts=[question], n_results=k)
    elapsed = time.perf_counter() - start
    hits = []
    for text, metadata, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append({
            "source": metadata["source"],
            "distance": distance,
            "preview": text.strip().replace("\n", " ")[:100],
        })
    return hits, elapsed


for question in comparison_questions:
    print("=" * 88)
    print("Question:", question)
    for item in embedding_comparison:
        hits, elapsed = retrieve_from_collection(item["collection"], question)
        print(f"\n{item['model']} ({item['dimensions']} dims, query {elapsed:.2f}s)")
        for rank, hit in enumerate(hits, start=1):
            print(f"  {rank}. {hit['source']} distance={hit['distance']:.4f} - {hit['preview']}...")

What to look for when presenting this:

- **Same top document:** both models understand the easy query; the smaller one may be good enough.
- **Different ranking:** the embedding model changed the retrieval result before the LLM saw anything.
- **Distance values:** compare distances only within the same model, not across models. Different embedding spaces produce different scales.
- **Latency and dimensions:** smaller models usually reduce embedding cost and memory, which matters when indexing many documents or serving many queries.

The outcome is empirical: choose the smallest embedding model that retrieves the right context reliably for your corpus and questions.

## 4. Retrieval only

At this point we have not asked the chat model to write anything. We are only searching the local knowledge base.

In [9]:
def retrieve(question: str, k: int = cfg.top_k) -> list[dict[str, Any]]:
    result = collection.query(query_texts=[question], n_results=k)
    hits = []
    for text, metadata, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append({
            "text": text,
            "source": metadata["source"],
            "distance": distance,
        })
    return hits


question = "What embedding model does this demo use and why?"
hits = retrieve(question)

print("Question:", question)
print("\nNearest documents:")
for index, hit in enumerate(hits, start=1):
    preview = hit["text"].strip().replace("\n", " ")[:120]
    print(f"{index}. {hit['source']} distance={hit['distance']:.4f} - {preview}...")

Question: What embedding model does this demo use and why?

Nearest documents:
1. qwen_models.md distance=0.5233 - # Qwen models used in this demo  The chat model in this demo is `qwen/qwen3.5-9b`, a 9-billion-parameter general-purpose...
2. lm_studio.md distance=0.6705 - # LM Studio local server  LM Studio is a desktop application that lets you download open-weight language models and serv...
3. rag_overview.md distance=0.7435 - # Retrieval-Augmented Generation (RAG)  Retrieval-Augmented Generation is a technique that grounds a language model in a...


## 5. Generation with retrieved context

Now we turn retrieval into RAG by placing the retrieved passages into the prompt. The model is instructed to answer only from that context and cite filenames.

In [10]:
def build_grounded_messages(question: str, hits: list[dict[str, Any]]) -> list[dict[str, str]]:
    context = "\n\n".join(f"[source: {hit['source']}]\n{hit['text']}" for hit in hits)
    return [
        {
            "role": "system",
            "content": (
                "You are a precise assistant. Answer using ONLY the provided context. "
                "If the answer is not in the context, say you do not know. "
                "Cite sources by filename."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


def answer_with_rag(question: str, k: int = cfg.top_k) -> str:
    hits = retrieve(question, k=k)
    response = client.chat.completions.create(
        model=cfg.chat_model,
        messages=build_grounded_messages(question, hits),
        temperature=0.2,
    )
    return response.choices[0].message.content or ""


answer = answer_with_rag(question)
display(Markdown(answer))



The demo uses the embedding model `text-embedding-qwen3-embedding-4b` [source: qwen_models.md]. It is used because it produces dense vectors for semantic similarity search over the document corpus, and the demos use the same embedding model for both stored documents and incoming questions so Chroma compares vectors from the same space [source: qwen_models.md].

## 6. The RAG pipeline as one function

This is the whole non-agentic RAG path: retrieve first, then ask the model. The application controls the retrieval step.

In [ ]:
for q in [
    "What is DevTalks and what is its official site?",
    "What are the three stages of RAG?",
    "What city hosted the first moon colony in this knowledge base?",
]:
    print("=" * 88)
    print("Question:", q)
    print(answer_with_rag(q))

## 7. Tool calling with simple Python functions

Tool calling changes who decides to run code. Instead of always retrieving first, we give the model tool schemas. The model can return a tool call, the application executes it, then the result goes back to the model.

In [ ]:
def add(a: int, b: int) -> int:
    return a + b


def current_time(timezone: str = "UTC") -> str:
    try:
        tz = ZoneInfo(timezone)
    except ZoneInfoNotFoundError:
        return f"Unknown timezone: {timezone!r}"
    return datetime.now(tz).isoformat(timespec="seconds")


basic_tool_functions = {
    "add": add,
    "current_time": current_time,
}

basic_tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Return the sum of two integers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "current_time",
            "description": "Return the current wall-clock time in an IANA timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "Example: UTC, Europe/Bucharest, America/New_York",
                    }
                },
                "required": ["timezone"],
            },
        },
    },
]

print(json.dumps(basic_tool_schemas, indent=2))

In [ ]:
def run_tool_calling_agent(
    question: str,
    tool_schemas: list[dict[str, Any]],
    tool_functions: dict[str, Any],
    system_prompt: str,
    max_steps: int = 5,
) -> str:
    messages: list[dict[str, Any]] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=cfg.chat_model,
            messages=messages,
            tools=tool_schemas,
            tool_choice="auto",
            temperature=0.2,
        )
        message = response.choices[0].message
        tool_calls = message.tool_calls or []

        if not tool_calls:
            return message.content or ""

        print(f"Step {step}: model requested {len(tool_calls)} tool call(s)")
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": call.id,
                    "type": "function",
                    "function": {
                        "name": call.function.name,
                        "arguments": call.function.arguments,
                    },
                }
                for call in tool_calls
            ],
        })

        for call in tool_calls:
            name = call.function.name
            try:
                arguments = json.loads(call.function.arguments or "{}")
            except json.JSONDecodeError:
                arguments = {}
            print(f"  calling {name}({arguments})")
            if name not in tool_functions:
                result = f"Unknown tool: {name}"
            else:
                result = tool_functions[name](**arguments)
            print(f"  result: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })

    return "Stopped before the model produced a final answer."


In [ ]:
tool_answer = run_tool_calling_agent(
    "What is 17 plus 25, and what time is it now in Europe/Bucharest?",
    basic_tool_schemas,
    basic_tool_functions,
    system_prompt=(
        "You are a concise assistant. Use tools when they are useful, "
        "then give a short natural-language answer."
    ),
)

display(Markdown(tool_answer))

## 8. RAG as a tool

Now the retrieval function becomes a tool named `search_knowledge`. This is the bridge to the MCP demo: MCP is one way to expose tools outside this notebook, but the tool-calling loop is the same idea.

In [ ]:
def search_knowledge(query: str, k: int = cfg.top_k) -> str:
    hits = retrieve(query, k=k)
    blocks = []
    for hit in hits:
        blocks.append(f"[source: {hit['source']}] (distance={hit['distance']:.4f})\n{hit['text'].strip()}")
    return "\n\n---\n\n".join(blocks)


rag_tool_functions = {"search_knowledge": search_knowledge}
rag_tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search the local DevTalks demo knowledge base and return relevant passages with source filenames.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural-language search query."},
                    "k": {"type": "integer", "description": "Number of passages to return.", "default": cfg.top_k},
                },
                "required": ["query"],
            },
        },
    }
]

print(search_knowledge("What are the three stages of RAG?", k=2)[:1000])

In [ ]:
rag_tool_answer = run_tool_calling_agent(
    "What is DevTalks, and what are the three stages of RAG? Cite sources.",
    rag_tool_schemas,
    rag_tool_functions,
    system_prompt=(
        "You are a grounded assistant. When the answer could be in the local knowledge base, "
        "call search_knowledge first. Answer using only retrieved passages and cite filenames. "
        "If the retrieved passages do not contain the answer, say you do not know."
    ),
)

display(Markdown(rag_tool_answer))